# Exploratory Data Analysis & Quantum-Enhanced Ablation Benchmark
### *Diabetic Retinopathy Tele-Screening Decision Support System*

This notebook visualizes the fundus image dataset, quality gate performance, 1280-d feature representations, and the 5-seed statistical ablation comparing **Hybrid Quantum (4-Qubit VQC)** vs **Matched Dense Control** vs **Classical Baseline**.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load internal manifest
manifest_path = Path('../data/manifests/demo_manifest.csv')
df = pd.read_csv(manifest_path)
print(f"Total manifest records: {len(df)}")
df.head()

## 1. Class Distribution Across Patient-Disjoint Splits

In [ ]:
split_counts = pd.crosstab(df['split'], df['label'])
split_counts.columns = ['Ungradable (-1)', 'No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'PDR (4)']
print(split_counts)

split_counts.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='viridis')
plt.title('Patient-Disjoint Split Label Distribution')
plt.ylabel('Image Count')
plt.xlabel('Dataset Split')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 2. Multi-Seed Quantum Ablation Results
Comparing parameter counts, Quadratic Weighted Kappa (QWK), Macro-F1, Referable DR Sensitivity, and Wilcoxon signed-rank test p-values.

In [ ]:
summary_path = Path('../reports/quantum_ablation_summary.json')
if summary_path.exists():
    with open(summary_path) as f:
        data = json.load(f)
    summary_df = pd.DataFrame(data['summary_table'])
    print("=== Summary Table ===")
    display_cols = ['head', 'total_params', 'latency_ms_per_image', 'accuracy_mean', 'macro_f1_mean', 'quadratic_weighted_kappa_mean', 'referable_sensitivity_mean', 'referable_specificity_mean']
    print(summary_df[display_cols].to_string(index=False))
    
    print("\n=== Wilcoxon Hypothesis Testing ===")
    print(json.dumps(data['statistical_tests'], indent=2))
else:
    print("Run python ablation.py first to generate benchmark metrics.")

## 3. Honest Clinical Reporting & Ethical Discussion
In accordance with strict clinical AI governance:
1. **No Benefit Presumption**: If the paired Wilcoxon test indicates $p > 0.05$ between the 4-qubit hybrid head and matched classical control, we report this candidly.
2. **Safety Invariants**: Ungradable fundus photos are flagged as `UNGRA_RECAPTURE` and NEVER allowed to emit a reassuring `No DR` output.
3. **Scope**: This system is a tele-screening prioritization aid and not a standalone diagnostic device.